# Track 2 · 03 Sealed-like evaluation — what we are testing, and how to zip it

**One job.** Score a few decoders for the PROTEUS BCI task on *sealed-like folds* (each fold holds out 2 people entirely, plus the later data of the other 8), show how each one learns, and package the two that can go to Codabench.

| model | evaluated | zip |
|---|---|---|
| REVE LoRA with causal session scaling | Step 2 (about 1 h) | Step 4b |
| covariance + LDA, 8-band filter bank, causal session reference | Step 3a (minutes) | Step 4a |
| PSD map + our small CNN / the paper's small CNN / ResNet-50 (pretrained) | Step 3b | none (evaluation only) |

Replaces `track2_02_eegnet.ipynb` (kept as the archive of everything tried before). Fake-data runs of this notebook are crash checks only.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

# What runs
RUN_REVE_FOLDS = True   # Step 2: REVE LoRA on the sealed-like folds (about an hour)
RUN_COV = True          # Step 3a: covariance + LDA, 8 bands
RUN_PSD = True          # Step 3b: PSD-map models (paper CNN, ResNet-50, our small CNN)
ZIP_COV = False         # Step 4a: fit on all 10 people, write and check the zip (needs RUN_COV)
ZIP_REVE = False        # Step 4b: one final REVE run on all 10 people, write and check the zip (about 10 min)
SEED = 33
LATE_RUNS = 2           # sealed-like folds: a single-session person is tested on their last LATE_RUNS runs

# REVE LoRA recipe (the v29 settings)
EPOCHS = 40
KEEP = 'last'           # the last epoch is kept; there is no held-out set to pick a best one from
LR_SCHEDULE = 'cosine'
BATCH = 64
LABEL_SMOOTHING = 0.12
WEIGHT_DECAY = 0.0
CROP_S = 2.0            # first 2 s of each 4 s window (1-3 s after the cue, before feedback)
HEAD_ONLY_EPOCHS = 1    # adapters stay frozen for this many epochs, the head learns first
LR_ADAPTER, LR_HEAD = 2e-4, 1e-3
LORA_RANK = 4           # attention adapters; alpha is 2 x rank
LORA_DROPOUT = 0.10
LORA_RANK_FFN = 8       # feed-forward adapters
LORA_ALPHA_FFN = 4
HEAD_SHRINK = 16        # only used if HEAD_DIMS is None
HEAD_DIMS = (128, 32)   # per-electrode funnel applied to each electrode's 512 REVE numbers
HEAD_DROPOUT_MID = 0.35
HEAD_NORM = True        # LayerNorm after every funnel linear layer
HEAD_INIT = 'kaiming'   # He starting weights for the funnel
HEAD_DROPOUT = 0.1
SESSION_N0 = 5.0        # prior weight (in windows) of the causal per-session power scale
SEALED_CV_SEEDS = 1
FINAL_SEEDS = 1         # Step 4b: seeds in the shipped REVE model (1 = one member, fastest)

# Training-time augmentation for REVE (set a value to 0 to turn it off)
AUG_MASK_RATIO = 0.15
AUG_MASK_NOISE = 1.0
AUG_MASK_RADIUS = 0.05
AUG_GAIN_SD = 0.1
AUG_SHIFT_S = 0.1

# Covariance + LDA and PSD models use the FULL 4 s windows and no REVE
CLASSICAL_N0 = 5        # prior weight (in windows) for the running per-session reference
PSD_EPOCHS = 60
PSD_SEEDS = 2
CHANNEL_ORDER = 'topographic'   # electrode order for the image models
RESNET_ARCH = 'resnet50'
RESNET_PRETRAINED = True
RESNET_EPOCHS = 15
RESNET_SEEDS = 1
RESNET_SIZE = 128

# Fixed plot limits, so curves from different runs lay side by side
PLOT_LOSS_YLIM = (0.0, 2.4)
PLOT_ACC_YLIM = (0.30, 0.70)

HEAD_TAG = '-'.join(map(str, HEAD_DIMS)) if HEAD_DIMS else str(HEAD_SHRINK)
print(f'NOTEBOOK VERSION: track2_03 sealed-like evaluation   REVE head {HEAD_TAG}   epochs {EPOCHS}   crop {CROP_S} s')

In [ ]:
%pip install -q 'braindecode==1.8.1' huggingface_hub safetensors

import importlib
for mod in ('braindecode', 'torch', 'numpy', 'sklearn'):
    print(f'{mod:12} {importlib.import_module(mod).__version__}')

In [ ]:
import json, os, time, warnings, zipfile
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch
from sklearn.metrics import balanced_accuracy_score

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

BASE = Path(ROOT) / 'Track2'
DATA = BASE / 'dreyer2026proteus'
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def load_npz(path, tries=5):
    """np.load that survives the Drive mount dropping mid-read (Errno 107): remount,
    then retry."""
    for attempt in range(tries):
        try:
            with np.load(path) as z:
                return {k: z[k] for k in z.files}
        except (OSError, zipfile.BadZipFile, EOFError) as e:
            if attempt == tries - 1 or not ROOT.startswith('/content/drive'):
                raise
            print(f'Drive read failed on {Path(path).name} ({e}); remounting and retrying')
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)


man = json.loads((DATA / 'manifest.json').read_text())
recs = man['recordings']
group = {r['subject']: r['group'] for r in recs.values()}
parts = {g: ([], [], [], [], []) for g in ('train', 'val', 'test')}   # X, y, person, session, run
CH = None
for name, r in sorted(recs.items()):
    z = load_npz(DATA / 'windows' / r['file'])
    p = parts[group[r['subject']]]
    p[0].append(z['X'].astype('float32')); p[1].append(z['y'])
    p[2].append(np.full(len(z['y']), r['subject']))
    p[3].append(np.full(len(z['y']), name))
    n_w = len(z['y'])      # windows are stored in run order; fall back to six equal chunks if no run ids
    p[4].append(np.asarray(z['run']) if 'run' in z else np.repeat(np.arange(1, 7), -(-n_w // 6))[:n_w])
    CH = CH or r['channels']
X, Y, P, S, RUNS = {}, {}, {}, {}, {}
for g, (xs, ys, ps, ss, rs) in parts.items():
    X[g], Y[g], P[g], S[g], RUNS[g] = (np.concatenate(xs), np.concatenate(ys), np.concatenate(ps),
                                       np.concatenate(ss), np.concatenate(rs))
N_CHANS, N_TIMES = X['train'].shape[1:]
N_CLASSES = 3
SFREQ = man['preprocessing']['sfreq']
XF = dict(X)      # full-length raw windows (4 s), kept for the covariance, PSD and streaming checks


def session_norm(Xg, Sg, prior_var, n0=5.0):
    """Causal per-session scaling. Windows of one session are contiguous and in run order. Each window
    is divided, per electrode, by the root of the mean power of the windows BEFORE it in its session
    (blended with a prior of n0 windows), after removing its own mean. Labels are never used."""
    out = np.empty(Xg.shape, dtype='float32')
    _, first = np.unique(Sg, return_index=True)
    for s_ in Sg[np.sort(first)]:
        idx = np.flatnonzero(Sg == s_)
        x = Xg[idx].astype('float64')
        x = x - x.mean(-1, keepdims=True)
        v = x.var(-1)                                   # (windows, channels)
        before = np.cumsum(v, 0) - v                    # power summed over earlier windows only
        t = np.arange(len(idx))[:, None]
        m = (n0 * prior_var[None] + before) / (n0 + t)
        out[idx] = (x / np.sqrt(m)[..., None]).astype('float32')
    return out


if CROP_S:
    # Windows on Drive are the harness's 4 s (1-5 s after the cue). The solver crops the
    # same way, because it already truncates inputs longer than the n_times it trained on.
    keep = int(round(CROP_S * SFREQ))
    assert keep <= N_TIMES, f'CROP_S={CROP_S} longer than the stored {N_TIMES} samples'
    X = {g: a[..., :keep] for g, a in X.items()}
    N_TIMES = keep
prior_var = np.concatenate([a.astype('float64').var(-1) for a in X.values()]).mean(0)   # per electrode
X = {g: session_norm(a, S[g], prior_var, SESSION_N0) for g, a in X.items()}   # what REVE reads
print('windows scaled by the running power of their own session (causal, no labels)')
assert not set(P['test']) & set(P['train']) and not set(P['val']) & set(P['train'])
for g in ('train', 'val', 'test'):
    print(f'{g:5}  {len(set(P[g])):2d} people  {len(Y[g]):6,d} windows   '
          f'mi {(Y[g] == 0).sum():,} / sub {(Y[g] == 1).sum():,} / word {(Y[g] == 2).sum():,}')
print(f'{N_CHANS} channels, {N_TIMES} samples at {SFREQ:g} Hz, device {DEVICE}')


def scores(y, pred):
    return dict(bal_acc=float(balanced_accuracy_score(y, pred)), acc=float((y == pred).mean()))


def null(y, pred, n=200):
    """Balanced accuracy with the predictions shuffled across windows: same predictions,
    no link to the labels. A model has learned something only if it beats this."""
    return np.array([balanced_accuracy_score(y, rng.permutation(pred)) for _ in range(n)])

In [ ]:
from braindecode.models import REVE
import torch.nn.functional as F


def standardize(x):
    # Windows arrive already scaled by the causal session norm, so this only converts the type.
    return torch.as_tensor(x).float()


def resample_fft(x, n_out):
    """Resample along time via the rfft spectrum (same function the solvers use)."""
    n_in = x.shape[-1]
    if n_in == n_out:
        return x
    spec = torch.fft.rfft(x.float(), dim=-1)
    out = torch.zeros(*spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device)
    keep = min(spec.shape[-1], out.shape[-1])
    out[..., :keep] = spec[..., :keep]
    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)


def load_reve(n_classes):
    """Pretrained brain-bzh/reve-base, built the way Track 1 builds it (200 Hz, 1 s
    patches). The task head is dropped; ours is added by the caller."""
    net = REVE.from_pretrained('brain-bzh/reve-base', n_outputs=n_classes, n_chans=N_CHANS,
                               n_times=200, sfreq=200.0, chs_info=[{'ch_name': c} for c in CH])
    net.final_layer = torch.nn.Identity()
    bank = {k.casefold() for k in net._position_bank.mapping}
    missing = [c for c in CH if c.casefold() not in bank]
    print(f'REVE pretrained loaded; electrodes without a known position: {missing or "none"}')
    assert not missing, 'REVE would mis-size its input; fix the channel names first'
    return net


class LoRALinear(torch.nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.drop = torch.nn.Dropout(dropout)    # adapter branch only; the frozen base is untouched
        for p_ in self.base.parameters():
            p_.requires_grad_(False)
        self.A = torch.nn.Parameter(torch.zeros(r, base.in_features))
        self.B = torch.nn.Parameter(torch.zeros(base.out_features, r))
        torch.nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale

    @property
    def weight(self):
        """Effective weight, W + scale * B A. braindecode >= 2.0 reads `net[1].weight` of the
        feed-forward GEGLU directly instead of calling the module, which would skip the
        adapter; with this property the adapter is applied there too. (The input dropout of
        this adapter does not apply on that path.)"""
        return self.base.weight + (self.B @ self.A) * self.scale


class PerElectrodeHead(torch.nn.Module):
    """Shrink each electrode's embedding with weights shared across electrodes (one linear
    layer, or a funnel of several), then a final layer with separate weights per electrode.
    norm: a LayerNorm after every funnel linear layer, which keeps the signal size steady.
    init: 'kaiming' gives the funnel layers He starting weights (PyTorch's default shrinks the
    signal at every layer)."""

    def __init__(self, n_chans, embed_dim, dims, n_outputs, dropout, mid_dropout=0.0,
                 norm=False, init='default'):
        super().__init__()
        dims = [dims] if isinstance(dims, int) else list(dims)
        self.norm = torch.nn.LayerNorm(embed_dim)
        if len(dims) == 1 and not norm:
            self.shrink = torch.nn.Linear(embed_dim, dims[0])        # keeps the older key names
        else:
            layers, prev = [], embed_dim
            for i, d in enumerate(dims):
                layers.append(torch.nn.Linear(prev, d))
                prev = d
                if norm:
                    layers.append(torch.nn.LayerNorm(d))
                if i < len(dims) - 1:
                    layers += [torch.nn.GELU(), torch.nn.Dropout(mid_dropout)]
            self.shrink = torch.nn.Sequential(*layers)
        if init == 'kaiming':
            for m in self.shrink.modules():
                if isinstance(m, torch.nn.Linear):
                    torch.nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                    torch.nn.init.zeros_(m.bias)
        self.drop = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(n_chans * dims[-1], n_outputs)

    def forward(self, tokens):                    # (B, chan, embed)
        z = self.drop(F.gelu(self.shrink(self.norm(tokens))))
        return self.out(z.flatten(1))


class ReveLoRA(torch.nn.Module):
    def __init__(self, n_classes, device, rank=LORA_RANK):
        super().__init__()
        self.net = load_reve(n_classes)
        self.net.final_layer = torch.nn.Identity()
        self.head = PerElectrodeHead(N_CHANS, self.net.embed_dim, HEAD_DIMS if HEAD_DIMS else HEAD_SHRINK,
                                     n_classes, HEAD_DROPOUT, HEAD_DROPOUT_MID, HEAD_NORM, HEAD_INIT)
        self.n_lora = 0
        for name, m in list(self.net.named_modules()):
            if isinstance(m, torch.nn.Linear) and name.split('.')[-1] in ('to_qkv', 'to_out'):
                parent = self.net.get_submodule(name.rsplit('.', 1)[0])
                setattr(parent, name.split('.')[-1], LoRALinear(m, rank, 2 * rank, LORA_DROPOUT))
                self.n_lora += 1
        self.n_ffn = 0
        if LORA_RANK_FFN:        # the two feed-forward matrices of each block, at net.1 and net.3
            for name, m in list(self.net.named_modules()):
                if (isinstance(m, torch.nn.Linear) and name.startswith('transformer.layers.')
                        and name.endswith(('.net.1', '.net.3'))):
                    parent = self.net.get_submodule(name.rsplit('.', 1)[0])
                    setattr(parent, name.split('.')[-1],
                            LoRALinear(m, LORA_RANK_FFN, LORA_ALPHA_FFN, LORA_DROPOUT))
                    self.n_ffn += 1
        for p_ in self.net.parameters():
            p_.requires_grad_(False)
        for n_, p_ in self.net.named_parameters():
            if n_.endswith(('.A', '.B')):
                p_.requires_grad_(True)
        self.to(device).eval()

    def forward(self, x):
        x = resample_fft(x.float(), int(round(x.shape[-1] * 200.0 / SFREQ)))   # REVE is 200 Hz
        feats = self.net(x, return_features=True)['features']      # (B, chan, patch, embed)
        return self.head(feats.mean(dim=2))                        # average the time patches only


def train_reve_lora(Xtr, ytr, Xv, yv, epochs, label, quiet=False, seed=SEED):
    torch.manual_seed(seed)
    net = ReveLoRA(N_CLASSES, DEVICE)
    adapters = [p_ for n_, p_ in net.net.named_parameters() if p_.requires_grad]
    opt = torch.optim.Adam([dict(params=adapters, lr=LR_ADAPTER),
                            dict(params=list(net.head.parameters()), lr=LR_HEAD)],
                           weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
             if LR_SCHEDULE == 'cosine' else None)
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    Xtr_t, ytr_t, Xv_t = torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xv)
    gen = torch.Generator().manual_seed(seed)

    @torch.no_grad()
    def probs(Xa):
        net.eval()
        Xa = torch.as_tensor(Xa)
        return torch.cat([net(standardize(Xa[i:i + 512].to(DEVICE))).softmax(-1)
                          for i in range(0, len(Xa), 512)]).cpu().numpy()

    if not quiet:
        n_ad = sum(p_.numel() for n_, p_ in net.net.named_parameters() if p_.requires_grad)
        print(f'{label}: REVE LoRA {n_ad:,} adapter params ({net.n_lora} attention + {net.n_ffn} feed-forward), '
              'per-electrode head, '
              f'{len(ytr):,} windows, lr {LR_SCHEDULE}, seed {seed}, head-only epochs {HEAD_ONLY_EPOCHS}, '
              f'lr adapters {LR_ADAPTER:g} / head {LR_HEAD:g}')
    best, best_state, hist = -1.0, None, []
    for ep in range(1, epochs + 1):
        for p_ in adapters:                      # staged: head alone first, then adapters too
            p_.requires_grad_(ep > HEAD_ONLY_EPOCHS)
        net.train()
        order = torch.randperm(len(ytr), generator=gen)
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(net(aug(standardize(Xtr_t[b].to(DEVICE)))), ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        lr_now = opt.param_groups[0]['lr']
        if sched is not None:
            sched.step()
        pv = probs(Xv_t)
        vl = float(-np.log(pv[np.arange(len(yv)), yv] + 1e-9).mean())
        sc = scores(yv, pv.argmax(1))
        hist.append(dict(epoch=ep, train_loss=tot / len(order), val_loss=vl, **sc))
        better = sc['bal_acc'] > best
        if better:
            best = sc['bal_acc']
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()
                          if k.endswith(('.A', '.B')) or k.startswith('head.')}
        if not quiet:
            print(f'  epoch {ep:2d}  train loss {tot / len(order):.4f}  val loss {vl:.4f}  '
                  f'val bal-acc {sc["bal_acc"]:.4f}  lr {lr_now:.1e}{"  *" if better else ""}  '
                  f'{time.time() - t0:.0f}s')
    if KEEP == 'best':
        net.load_state_dict(best_state, strict=False)
    return net, hist, probs


print('ready')

In [ ]:
# Source of truth for the mask generator is src/eeg_masking.py in the repo.
import numpy as np


class ElectrodeMasker:
    """positions: (C, 3) electrode coordinates in metres (REVE's position bank is in metres,
    head radius about 0.095). n_time: number of time tokens per window."""

    def __init__(self, positions, n_time=1, ratio=0.2, ratio_concentration=8.0,
                 spatial_scale=0.05, temporal_scale=1.0, scale_sigma=0.5,
                 n_bumps=(2, 5), noise=0.3, min_visible_frac=0.4, balance=True):
        pos = np.asarray(positions, dtype=float)
        self.C, self.T = len(pos), int(n_time)
        self.D = np.linalg.norm(pos[:, None] - pos[None], axis=-1)       # (C, C) metres
        self.ratio, self.conc = float(ratio), float(ratio_concentration)
        self.sp, self.tp, self.sig = float(spatial_scale), float(temporal_scale), float(scale_sigma)
        self.n_bumps, self.noise = n_bumps, float(noise)
        self.min_visible = max(1, int(round(min_visible_frac * self.C)))
        if balance:
            density = np.exp(-(self.D ** 2) / (2 * self.sp ** 2)).sum(1)
            w = 1.0 / density
        else:
            w = np.ones(self.C)
        self.centre_p = w / w.sum()

    def _one(self, rng):
        C, T = self.C, self.T
        a, b = self.ratio * self.conc, (1 - self.ratio) * self.conc
        ratio = float(rng.beta(a, b)) if 0 < self.ratio < 1 else self.ratio
        field = np.zeros((C, T))
        t_axis = np.arange(T)
        for _ in range(int(rng.integers(self.n_bumps[0], self.n_bumps[1] + 1))):
            c = rng.choice(C, p=self.centre_p)
            r = self.sp * np.exp(self.sig * rng.standard_normal())
            L = self.tp * np.exp(self.sig * rng.standard_normal())
            t0 = rng.uniform(-0.5, T - 0.5)
            amp = rng.uniform(0.5, 1.5)
            field += amp * np.exp(-self.D[c] ** 2 / (2 * r ** 2))[:, None] \
                * np.exp(-((t_axis - t0) ** 2) / (2 * L ** 2))[None]
        field = (field - field.mean()) / (field.std() + 1e-9)
        field = field + self.noise * rng.standard_normal((C, T)) + 1e-6 * rng.standard_normal((C, T))
        k = int(round(ratio * C * T))
        k = min(k, (C - self.min_visible) * T)
        mask = np.zeros(C * T, dtype=bool)
        mask[np.argsort(-field.ravel())[:k]] = True
        mask = mask.reshape(C, T)
        # every time token keeps at least `min_visible` electrodes
        for t in range(T):
            over = mask[:, t].sum() - (C - self.min_visible)
            if over > 0:
                hidden = np.flatnonzero(mask[:, t])
                mask[hidden[np.argsort(field[hidden, t])[:over]], t] = False
        return mask

    def __call__(self, batch_size, rng=None):
        """(B, C, T) boolean mask, True = hidden. Pass a numpy Generator for repeatability."""
        rng = rng if rng is not None else np.random.default_rng()
        return np.stack([self._one(rng) for _ in range(batch_size)])


def expand_mask(mask, patch_len, n_samples):
    """(B, C, T) token mask -> (B, C, n_samples) sample mask, each token covering patch_len."""
    out = np.repeat(mask, patch_len, axis=-1)[..., :n_samples]
    if out.shape[-1] < n_samples:                       # pad with the last token's value
        out = np.concatenate([out, np.repeat(out[..., -1:], n_samples - out.shape[-1], -1)], -1)
    return out


AUG_ON = AUG_MASK_RATIO > 0 or AUG_GAIN_SD > 0 or AUG_SHIFT_S > 0
NEEDS_TRAINING = RUN_REVE_FOLDS or ZIP_REVE    # the covariance and PSD models do not need REVE
if AUG_MASK_RATIO > 0 and NEEDS_TRAINING:
    _tmp = load_reve(N_CLASSES)                    # only to read REVE's electrode positions
    _folded = {k.casefold(): k for k in _tmp._position_bank.mapping}
    POS = _tmp.get_positions([_folded[c.casefold()] for c in CH]).detach().cpu().numpy().astype(float)
    del _tmp
    TOK = int(SFREQ)                               # one time token = 1 s
    MASKER = ElectrodeMasker(POS, n_time=max(1, -(-N_TIMES // TOK)), ratio=AUG_MASK_RATIO,
                             spatial_scale=AUG_MASK_RADIUS, noise=AUG_MASK_NOISE)


def aug(x):
    """Applied to a standardised training batch (B, C, T). Hidden electrodes become zeros,
    which is also how the solver treats an electrode that is absent at inference."""
    if not AUG_ON:
        return x
    B, C, T = x.shape
    g = np.random.default_rng(int(torch.randint(0, 2 ** 31 - 1, (1,))))   # follows torch's seed
    if AUG_GAIN_SD > 0:
        x = x * torch.exp(AUG_GAIN_SD * torch.randn(B, C, 1, device=x.device))
    if AUG_SHIFT_S > 0:
        s = int(round(AUG_SHIFT_S * SFREQ))
        sh = g.integers(-s, s + 1, B).tolist()
        x = torch.stack([torch.roll(x[i], sh[i], dims=-1) for i in range(B)])
    if AUG_MASK_RATIO > 0:
        m = expand_mask(MASKER(B, g), TOK, T)
        x = x.masked_fill(torch.as_tensor(m, device=x.device), 0.0)
    return x


print(f'augmentation: mask {AUG_MASK_RATIO} (noise {AUG_MASK_NOISE}, radius {AUG_MASK_RADIUS} m), '
      f'gain sd {AUG_GAIN_SD}, shift {AUG_SHIFT_S} s' if AUG_ON else 'augmentation off')

## Sealed-like folds (shared setup)

The sealed set is later sessions of the 10 public people plus 10 unseen people. Each fold holds out 2 people entirely (**unseen**) and the later data of the other 8 (**seen**); training uses only earlier data. MIX is the 50/50 average. The seen half is optimistic (same-day later runs) and the true mix is unknown, so read UNSEEN first.

In [ ]:
# Shared by Steps 2, 3a and 3b: the sealed-like folds, so every method is scored on exactly the same data.
RESULTS = {}      # method name -> (unseen, seen-later, mix)
HIST = {}         # method name -> list of per-epoch histories (one per fold and seed): train loss, held-out loss, accuracy
POOLED = {}       # method name -> held-out balanced accuracy pooled over all held-out windows, per fold
FIT_TRAIN = {}    # method name -> balanced accuracy on its own training windows, per fold (for models without epochs)
import matplotlib.pyplot as plt


def sealed_setup():
    people = sorted(set(P['train']) | set(P['val']) | set(P['test']))
    cat = lambda d: np.concatenate([d[g] for g in ('train', 'val', 'test')])
    sd = dict(Xa=cat(X), XFa=cat(XF), Ya=cat(Y), Pa=cat(P), Sa=cat(S), Ra=cat(RUNS))
    ses_no = np.array([int(str(s_).split('ses-')[1]) for s_ in sd['Sa']])
    later = np.zeros(len(sd['Ya']), bool)         # the data a represented person is tested on
    for p_ in people:
        m_ = sd['Pa'] == p_
        sess_ = sorted(set(ses_no[m_]))
        if len(sess_) > 1:
            later |= m_ & (ses_no > sess_[0])
        else:
            later |= m_ & np.isin(sd['Ra'], sorted(set(sd['Ra'][m_]))[-LATE_RUNS:])
    order = np.random.default_rng(SEED).permutation(people)
    sd.update(later=later, folds=[order[i::5] for i in range(5)])
    return sd


SD = sealed_setup()
_, _first = np.unique(SD['Sa'], return_index=True)
SESSIONS = [np.flatnonzero(SD['Sa'] == s_) for s_ in SD['Sa'][np.sort(_first)]]   # windows of each session, in order


def _cell_scores(rows, pred_of):
    out = []
    for p_, s_ in sorted({(SD['Pa'][r], SD['Sa'][r]) for r in rows}):
        idx = [r for r in rows if SD['Pa'][r] == p_ and SD['Sa'][r] == s_]
        out.append(float(balanced_accuracy_score(SD['Ya'][idx], pred_of[idx])))
    return out


def sealed_eval(name, fit_predict, verbose=True):
    # fit_predict(train_rows, eval_rows) -> class probabilities (len(eval_rows), 3) for the eval rows.
    Pa, later = SD['Pa'], SD['later']
    unseen_all, seen_all, mix_all = [], [], []
    SD['name'] = name
    for f, held in enumerate(SD['folds']):
        unseen = np.isin(Pa, held)
        seen_late = ~unseen & later
        train_m = ~unseen & ~later
        ev = unseen | seen_late
        probs = fit_predict(np.flatnonzero(train_m), np.flatnonzero(ev))
        pred_of = np.full(len(Pa), -1)
        pred_of[np.flatnonzero(ev)] = np.asarray(probs).argmax(1)
        POOLED.setdefault(name, []).append(float(balanced_accuracy_score(SD['Ya'][ev], pred_of[ev])))
        u = _cell_scores(list(np.flatnonzero(unseen)), pred_of)
        sn = _cell_scores(list(np.flatnonzero(seen_late)), pred_of)
        mix = 0.5 * (np.mean(u) + np.mean(sn))
        unseen_all.append(np.mean(u)); seen_all.append(np.mean(sn)); mix_all.append(mix)
        if verbose:
            print(f'  [{name}] fold {f + 1}/5  unseen {np.mean(u):.4f} ({len(u)} cells)   '
                  f'seen-later {np.mean(sn):.4f} ({len(sn)} cells)   mix {mix:.4f}   '
                  f'(trained on {int(train_m.sum())} windows)')
    RESULTS[name] = (float(np.mean(unseen_all)), float(np.mean(seen_all)), float(np.mean(mix_all)))
    return RESULTS[name]


def print_results():
    print('\nsealed-like folds, mean over 5 folds (chance 0.333)')
    print(f'  {"method":58s} {"unseen":>7s} {"seen":>7s} {"MIX":>7s}')
    for k, (u, s_, m_) in RESULTS.items():
        print(f'  {k:58s} {u:7.4f} {s_:7.4f} {m_:7.4f}')


def plot_curves(names, fname='curves_sealed_like.png'):
    # Mean over folds (and seeds) of the per-epoch training loss, held-out loss and held-out accuracy, with the
    # min-max band across folds. Held-out = the unseen people and the later data of the seen people, pooled.
    # Axis limits are fixed (PLOT_LOSS_YLIM, PLOT_ACC_YLIM) so figures from different runs compare.
    names = [n for n in names if HIST.get(n)]
    if not names:
        print('no learning curves were recorded'); return
    fig, axes = plt.subplots(2, len(names), figsize=(4.8 * len(names), 7.2), squeeze=False)
    for j, nm in enumerate(names):
        runs = HIST[nm]
        n_ep = min(len(r) for r in runs)
        ep = np.arange(1, n_ep + 1)
        arr = lambda key: np.array([[h_[key] for h_ in r[:n_ep]] for r in runs])
        tl, vl, ba = arr('train_loss'), arr('val_loss'), arr('bal_acc')
        ax = axes[0, j]
        for a_, lab in ((tl, 'train loss (augmented, smoothed)'), (vl, 'held-out loss')):
            ax.plot(ep, a_.mean(0), label=lab)
            ax.fill_between(ep, a_.min(0), a_.max(0), alpha=0.15)
        ax.axhline(float(np.log(3)), ls='--', c='grey', lw=1, label='chance loss')
        ax.set_ylim(*PLOT_LOSS_YLIM); ax.grid(alpha=0.3); ax.set_xlabel('epoch')
        ax.set_title(f'{nm[:42]}\nfinal gap {vl.mean(0)[-1] - tl.mean(0)[-1]:+.2f}', fontsize=8)
        if j == 0:
            ax.set_ylabel('loss'); ax.legend(fontsize=7, loc='upper right')
        ax = axes[1, j]
        ax.plot(ep, ba.mean(0), c='C2', label='held-out balanced accuracy')
        ax.fill_between(ep, ba.min(0), ba.max(0), color='C2', alpha=0.15)
        ax.axhline(1 / 3, ls='--', c='grey', lw=1, label='chance')
        ax.set_ylim(*PLOT_ACC_YLIM); ax.grid(alpha=0.3); ax.set_xlabel('epoch')
        if j == 0:
            ax.set_ylabel('balanced accuracy'); ax.legend(fontsize=7, loc='upper left')
    fig.tight_layout()
    fig.savefig(BASE / 'weights' / fname, dpi=120, bbox_inches='tight')
    plt.show()

## Step 2 — REVE LoRA on the sealed-like folds

Pretrained REVE with attention and feed-forward LoRA adapters, a per-electrode funnel head, causal session scaling. Curves show training loss (augmented, label-smoothed), held-out loss and held-out balanced accuracy per epoch.

In [ ]:
if RUN_REVE_FOLDS:
    Xa_, Ya_ = SD['Xa'], SD['Ya']

    def reve_fit_predict(tr, ev):
        probs_seeds = []
        for s_ in range(SEALED_CV_SEEDS):
            net_f, hist_f, probs_f = train_reve_lora(Xa_[tr], Ya_[tr], Xa_[ev], Ya_[ev], EPOCHS,
                                              'sealed-like fold', quiet=True, seed=SEED + s_)
            HIST.setdefault(SD['name'], []).append(hist_f)
            probs_seeds.append(probs_f(Xa_[ev]))
            del net_f
        return np.mean(probs_seeds, 0)

    keep_before, KEEP = KEEP, 'last'
    REVE_NAME = 'REVE LoRA, causal session scaling'
    sealed_eval(REVE_NAME, reve_fit_predict)
    KEEP = keep_before
    print_results()
    plot_curves([REVE_NAME], 'curves_reve.png')
else:
    print('Step 2 skipped (RUN_REVE_FOLDS = False)')

## Step 3a — covariance, tangent space, LDA (no REVE, no GPU)

Each full 4 s window becomes one OAS-shrunk covariance matrix per band of an 8-band filter bank (the covariance of the power spectrum, one cross-spectral matrix per band), whitened against the causal session reference, mapped to the tangent space and classified by shrinkage LDA.

In [ ]:
if RUN_COV:
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as _LDA

    BAND = {}
    # A fine filter bank is the practical form of a "covariance of the power spectrum": one covariance matrix per
    # narrow band is a cross-spectral matrix (how electrodes co-vary at that frequency).
    FB8 = [f'fb{j + 1}' for j in range(8)]
    for j_, (lo_, hi_) in enumerate([(4, 7), (7, 9), (9, 11), (11, 13), (13, 16), (16, 20), (20, 26), (26, 35)]):
        BAND[FB8[j_]] = (float(lo_), float(hi_))

    def band_filter(x, lo, hi):                  # x: (N, C, T) float64 torch; zero out other frequencies
        T_ = x.shape[-1]
        F_ = torch.fft.rfft(x, dim=-1)
        f_ = torch.fft.rfftfreq(T_, d=1.0 / SFREQ)
        return torch.fft.irfft(F_ * ((f_ >= lo) & (f_ <= hi)).to(x.dtype), n=T_, dim=-1)

    def oas_cov(x):                              # (N, C, T) -> (N, C, C), shrunk toward a scaled identity
        N_, C_, T_ = x.shape
        x = x - x.mean(-1, keepdim=True)
        S_ = x @ x.transpose(1, 2) / T_
        alpha = (S_ ** 2).mean((1, 2))
        mu = S_.diagonal(dim1=1, dim2=2).sum(-1) / C_
        rho = torch.clamp((alpha + mu ** 2) / ((T_ + 1.0) * (alpha - mu ** 2 / C_)), max=1.0)
        return (1 - rho)[:, None, None] * S_ + (rho * mu)[:, None, None] * torch.eye(C_, dtype=x.dtype)

    def sym_fn(M, fn):                           # apply fn to the eigenvalues of symmetric matrices
        w_, V_ = torch.linalg.eigh(M)
        return (V_ * fn(w_).unsqueeze(-2)) @ V_.transpose(-1, -2)

    def tangent_vec(Chat):                       # SPD (N, C, C) -> (N, C(C+1)/2); off-diagonals x sqrt(2)
        C_ = Chat.shape[-1]
        L_ = sym_fn(Chat, torch.log)
        iu = torch.triu_indices(C_, C_)
        return L_[:, iu[0], iu[1]] * torch.where(iu[0] == iu[1], 1.0, 2.0 ** 0.5)

    XF64 = torch.as_tensor(SD['XFa'], dtype=torch.float64)
    COV, LOGCOV = {}, {}
    for b_, (lo_, hi_) in BAND.items():
        COV[b_] = torch.cat([oas_cov(band_filter(XF64[i:i + 400], lo_, hi_)) for i in range(0, len(XF64), 400)])
        LOGCOV[b_] = sym_fn(COV[b_], torch.log)
    del XF64
    def band_features(b_, tr, mode):
        Cs, Ls = COV[b_], LOGCOV[b_]
        prior = Ls[tr].mean(0)                   # log-Euclidean mean of the training windows
        if mode == 'global':                     # one fixed reference for everybody
            A_ = sym_fn(prior, lambda w_: torch.exp(-w_ / 2))
        else:
            Lref = torch.empty_like(Ls)
            for idx in SESSIONS:
                Ls_s = Ls[idx]
                if mode == 'session':            # oracle: the whole session (uses the future; not causal)
                    Lref[idx] = Ls_s.mean(0)
                else:                            # causal: windows BEFORE this one, blended with the prior
                    before = torch.cumsum(Ls_s, 0) - Ls_s
                    t_ = torch.arange(len(idx), dtype=Ls.dtype).view(-1, 1, 1)
                    Lref[idx] = (CLASSICAL_N0 * prior + before) / (CLASSICAL_N0 + t_)
            A_ = sym_fn(Lref, lambda w_: torch.exp(-w_ / 2))
        return tangent_vec(A_ @ Cs @ A_)

    def make_cov_fit_predict(bands, mode):
        def fit_predict(tr, ev):
            F_ = torch.cat([band_features(b_, tr, mode) for b_ in bands], 1).numpy()
            clf = _LDA(solver='lsqr', shrinkage='auto').fit(F_[tr], SD['Ya'][tr])
            FIT_TRAIN.setdefault(SD['name'], []).append(float(balanced_accuracy_score(SD['Ya'][tr], clf.predict(F_[tr]))))
            return clf.predict_proba(F_[ev])
        return fit_predict

    sealed_eval('covariance + LDA, 8-band filter bank, causal session reference', make_cov_fit_predict(FB8, 'causal'), verbose=False)
    print_results()
    print('\nfit check (no epochs here: the gap between the two numbers is the overfitting)')
    for k_ in FIT_TRAIN:
        print(f'  {k_[:60]:60s} train {np.mean(FIT_TRAIN[k_]):.3f}   held-out (pooled) {np.mean(POOLED[k_]):.3f}')
else:
    print('Step 3a skipped (RUN_COV = False)')

## Step 3b — PSD-map models

A log-PSD map (electrode x frequency, Welch, 2-40 Hz) relative to the session's own causal running baseline, read by: the paper's small CNN (arXiv 2202.03265, Table 1), a pretrained ResNet-50 (the paper's Res-PSD) and our small EEGNet-style CNN. Evaluation only, no zip.

In [ ]:
if RUN_PSD:
    import torchvision
    nn_ = torch.nn

    # ---- electrode order for the image models: a CNN reads neighbouring rows, so put neighbours next to each other
    def topographic_order(names):
        try:
            import mne
            pos = mne.channels.make_standard_montage('standard_1005').get_positions()['ch_pos']
            low = {k.casefold(): v for k, v in pos.items()}
            xy = [low[n.casefold()][:2] for n in names]
        except Exception as e_:
            print('electrode positions unavailable, keeping the given order:', e_)
            return np.arange(len(names))
        x_, y_ = np.array([p[0] for p in xy]), np.array([p[1] for p in xy])
        row = np.round(y_ / 0.03).astype(int)               # rows about 3 cm apart, front to back
        return np.lexsort((x_, -row))                       # then left to right within a row

    EORD = topographic_order(CH) if CHANNEL_ORDER == 'topographic' else np.arange(len(CH))
    print('electrode order for the image models:', [CH[i] for i in EORD])

    # ---- inputs, computed once (no labels used)
    def log_psd(x, seg=120, step=60, fmin=2.0, fmax=40.0):      # (N, C, T) -> (N, C, F) log10 power
        x = torch.as_tensor(x, dtype=torch.float64)
        x = x - x.mean(-1, keepdim=True)
        win = torch.hann_window(seg, dtype=torch.float64)
        spec = torch.fft.rfft(x.unfold(-1, seg, step) * win, dim=-1).abs() ** 2
        P_ = spec.mean(2) / (win ** 2).sum()
        f_ = torch.fft.rfftfreq(seg, d=1.0 / SFREQ)
        keep_f = (f_ >= fmin) & (f_ <= fmax)
        return torch.log10(P_[..., keep_f] + 1e-12).float().numpy()

    PSD = np.concatenate([log_psd(SD['XFa'][i:i + 400]) for i in range(0, len(SD['XFa']), 400)])[:, EORD]

    SESSIONS_PSD = SESSIONS

    def psd_causal_baseline(Z, prior):           # subtract the running mean of EARLIER windows of the session
        out = np.empty_like(Z)
        for idx in SESSIONS_PSD:
            z_ = Z[idx]
            before = np.cumsum(z_, 0) - z_
            t_ = np.arange(len(idx)).reshape(-1, 1, 1)
            out[idx] = z_ - (CLASSICAL_N0 * prior + before) / (CLASSICAL_N0 + t_)
        return out

    # ---- models
    class PsdCNN(nn_.Module):                    # EEGNet-style over a (electrode x frequency) power map
        def __init__(self, n_chans, n_freq, n_classes=3, f1=8, d=2, drop=0.4):
            super().__init__()
            self.spectral = nn_.Sequential(nn_.Conv2d(1, f1, (1, 5), padding=(0, 2), bias=False), nn_.BatchNorm2d(f1))
            self.spatial = nn_.Sequential(nn_.Conv2d(f1, f1 * d, (n_chans, 1), groups=f1, bias=False),
                                          nn_.BatchNorm2d(f1 * d), nn_.ELU(), nn_.AvgPool2d((1, 3)), nn_.Dropout(drop))
            self.sep = nn_.Sequential(nn_.Conv2d(f1 * d, f1 * d, (1, 5), padding=(0, 2), groups=f1 * d, bias=False),
                                      nn_.Conv2d(f1 * d, f1 * d, 1, bias=False), nn_.BatchNorm2d(f1 * d),
                                      nn_.ELU(), nn_.AvgPool2d((1, 2)), nn_.Dropout(drop))
            with torch.no_grad():
                n_out = self.sep(self.spatial(self.spectral(torch.zeros(1, 1, n_chans, n_freq)))).numel()
            self.fc = nn_.Linear(n_out, n_classes)

        def forward(self, x):                    # x: (B, C, F)
            return self.fc(self.sep(self.spatial(self.spectral(x.unsqueeze(1)))).flatten(1))

    class PaperCNN(nn_.Module):                  # Table 1 of Ramirez-Aristizabal et al. 2022: three 4x4 stride-2 convs, GAP, FC; here on the PSD map
        def __init__(self, n_classes=3):
            super().__init__()
            blk = lambda i, o: nn_.Sequential(nn_.Conv2d(i, o, 4, stride=2, padding=1), nn_.BatchNorm2d(o), nn_.ReLU())
            self.body = nn_.Sequential(blk(1, 32), blk(32, 64), blk(64, 128), nn_.AdaptiveAvgPool2d(1), nn_.Flatten())
            self.head = nn_.Sequential(nn_.Linear(128, 128), nn_.BatchNorm1d(128), nn_.ReLU(), nn_.Dropout(0.3),
                                       nn_.Linear(128, n_classes))

        def forward(self, x):                    # x: (B, C, samples), an image with electrodes as rows
            return self.head(self.body(x.unsqueeze(1)))

    class ResNetImg(nn_.Module):                 # ImageNet ResNet on the image stacked three times as RGB
        def __init__(self, arch, pretrained, size, n_classes=3):
            super().__init__()
            m = getattr(torchvision.models, arch)(weights='IMAGENET1K_V1' if pretrained else None)
            m.fc = nn_.Linear(m.fc.in_features, n_classes)
            self.m, self.size = m, size
            self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
            self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        def forward(self, x):                    # x: (B, H, W) in [0, 1]
            x = F.interpolate(x.unsqueeze(1), size=(self.size, self.size), mode='bilinear', align_corners=False)
            return self.m((x.repeat(1, 3, 1, 1) - self.mean) / self.std)

    def train_img(make_net, Ztr, ytr, seed, epochs, lr, mid, Zev=None, yev=None, batch=32):
        torch.manual_seed(seed)
        net = make_net().to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-2)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
        Xt, yt = torch.tensor(Ztr, dtype=torch.float32), torch.tensor(ytr)
        gen = torch.Generator().manual_seed(seed)
        hist = []
        for ep in range(epochs):
            net.train()
            tot, cnt = 0.0, 0
            order = torch.randperm(len(yt), generator=gen)
            for i in range(0, len(order), batch):
                b = order[i:i + batch]
                if len(b) < 2:
                    continue
                x = Xt[b].to(DEVICE)
                keep_rows = (torch.rand(len(b), x.shape[1], 1, device=x.device) > 0.1).float()   # drop 10% of electrodes
                x = x * keep_rows + mid * (1 - keep_rows)
                x = x + 0.05 * (x.std() + 1e-6) * torch.randn_like(x)
                opt.zero_grad()
                loss_ = loss_fn(net(x), yt[b].to(DEVICE))
                loss_.backward()
                opt.step()
                tot += float(loss_) * len(b); cnt += len(b)
            sched.step()
            if Zev is not None:               # held-out loss and accuracy after every epoch, to see under- or over-fitting
                p_ = predict_img(net.eval(), Zev)
                hist.append(dict(epoch=ep + 1, train_loss=tot / max(cnt, 1),
                                 val_loss=float(-np.log(p_[np.arange(len(yev)), yev] + 1e-9).mean()),
                                 bal_acc=float(balanced_accuracy_score(yev, p_.argmax(1)))))
        return net.eval(), hist

    def predict_img(net, Z):
        out = []
        with torch.no_grad():
            for i in range(0, len(Z), 128):
                out.append(torch.softmax(net(torch.tensor(Z[i:i + 128], dtype=torch.float32).to(DEVICE)), -1).cpu().numpy())
        return np.concatenate(out)

    def make_img_fit_predict(kind):
        def fit_predict(tr, ev):
            Zn = psd_causal_baseline(PSD, PSD[tr].mean(0))   # power relative to the session's own running baseline
            Zn = Zn / Zn[tr].std()
            if kind == 'psd_small':
                Z, make, lr, ep, mid, seeds = Zn, (lambda: PsdCNN(Zn.shape[1], Zn.shape[2])), 2e-3, PSD_EPOCHS, 0.0, PSD_SEEDS
            elif kind == 'psd_paper':
                Z, make, lr, ep, mid, seeds = Zn, PaperCNN, 2e-3, PSD_EPOCHS, 0.0, PSD_SEEDS
            else:                                         # psd_resnet
                Z = (np.clip(Zn, -3, 3) + 3) / 6
                make, lr, ep, mid, seeds = (lambda: ResNetImg(RESNET_ARCH, RESNET_PRETRAINED, RESNET_SIZE)), 1e-4, RESNET_EPOCHS, 0.5, RESNET_SEEDS
            probs = []
            for s_ in range(seeds):
                net_, hist_ = train_img(make, Z[tr], SD['Ya'][tr], SEED + s_, ep, lr, mid, Zev=Z[ev], yev=SD['Ya'][ev])
                HIST.setdefault(SD['name'], []).append(hist_)
                probs.append(predict_img(net_, Z[ev]))
            return np.mean(probs, 0)
        return fit_predict

    for label_, kind_ in [
            ('PSD map + our small CNN (causal session baseline)', 'psd_small'),
            ('PSD map + the paper small CNN (causal session baseline)', 'psd_paper'),
            (f'{RESNET_ARCH} on the PSD map, paper Res-PSD (pretrained={RESNET_PRETRAINED})', 'psd_resnet')]:
        sealed_eval(label_, make_img_fit_predict(kind_), verbose=False)
        print_results()
    plot_curves([k_ for k_ in HIST if 'PSD map' in k_ or 'PSD' in k_], 'curves_psd_models.png')
else:
    print('Step 3b skipped (RUN_PSD = False)')

## Step 4a — package the covariance + LDA model

Fits the classifier on **all** 10 people and all sessions with the 8 bands, writes `submission.py` + weights, loads the folder the way the harness does and streams windows one at a time. The zip is only written if every check passes. Nothing is submitted unless you upload it yourself.

In [ ]:
# Source of truth: solvers/track2_bci_covlda.py in the repo.
SUBMISSION_PY_COV = '"""Track 02 submission: covariance -> tangent space -> LDA, with a causal per-session reference.\n\nNo pretrained network and no GPU. For every window the model:\n  1. matches incoming electrodes to the trained ones by name (extras ignored, missing ones zero-filled),\n  2. band-filters the full window (FFT mask) in each band of the config,\n  3. computes an OAS-shrunk electrode covariance matrix per band,\n  4. whitens it against a reference covariance and takes the matrix log (the tangent space),\n     listing the upper triangle (off-diagonals x sqrt(2)),\n  5. applies a linear classifier (the exported LDA weights) and returns the arg-max class.\n\nThe reference is causal: the log-Euclidean mean of the windows BEFORE this one in the same session,\nblended with the training prior (weight n0 windows). The only state is a running sum of log-covariances\nand a window count; `reset_state` clears it, and the harness calls it at the start of every session.\n\nThe feature code matches `track2_03_sealed_eval.ipynb` (Step 2d); keep the two in step. Everything runs on CPU\nin float64: a 41 x 41 eigendecomposition is faster there than a GPU round trip.\n"""\n\nimport json\n\nimport torch\n\nfrom benchmark_utils.base_solver import CompetSolver\n\n\ndef resample_fft(x, n_out):\n    """Resample along time via the rfft spectrum. Pure torch: nothing is installed at evaluation."""\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x, dim=-1)\n    out = torch.zeros(*spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype)\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\ndef band_filter(x, lo, hi, fs):\n    """x: (N, C, T). Zero every frequency outside [lo, hi]."""\n    T = x.shape[-1]\n    spec = torch.fft.rfft(x, dim=-1)\n    f = torch.fft.rfftfreq(T, d=1.0 / fs)\n    return torch.fft.irfft(spec * ((f >= lo) & (f <= hi)).to(x.dtype), n=T, dim=-1)\n\n\ndef oas_cov(x):\n    """(N, C, T) -> (N, C, C) covariance, shrunk toward a scaled identity (OAS)."""\n    _, C, T = x.shape\n    x = x - x.mean(-1, keepdim=True)\n    S = x @ x.transpose(1, 2) / T\n    alpha = (S ** 2).mean((1, 2))\n    mu = S.diagonal(dim1=1, dim2=2).sum(-1) / C\n    rho = torch.clamp((alpha + mu ** 2) / ((T + 1.0) * (alpha - mu ** 2 / C)), max=1.0)\n    return (1 - rho)[:, None, None] * S + (rho * mu)[:, None, None] * torch.eye(C, dtype=x.dtype)\n\n\ndef sym_fn(M, fn):\n    """Apply fn to the eigenvalues of symmetric matrices."""\n    w, V = torch.linalg.eigh(M)\n    return (V * fn(w).unsqueeze(-2)) @ V.transpose(-1, -2)\n\n\ndef tangent_vec(Chat):\n    """SPD (N, C, C) -> (N, C(C+1)/2): matrix log, upper triangle, off-diagonals x sqrt(2)."""\n    C = Chat.shape[-1]\n    L = sym_fn(Chat, torch.log)\n    iu = torch.triu_indices(C, C)\n    return L[:, iu[0], iu[1]] * torch.where(iu[0] == iu[1], 1.0, 2.0 ** 0.5)\n\n\nclass CovLDA:\n\n    def __init__(self, cfg, state, in_names, in_sfreq):\n        self.cfg = cfg\n        self.bands = [(b[0], float(b[1]), float(b[2])) for b in cfg["bands"]]\n        self.n0 = float(cfg["n0"])\n        self.fs = float(cfg["sfreq"])\n        self.in_sfreq = float(in_sfreq)\n        self.prior = {name: state[f"prior_{name}"].double() for name, _, _ in self.bands}\n        self.coef = state["coef"].double()                      # (classes, features)\n        self.intercept = state["intercept"].double()\n        lookup = {n.casefold(): i for i, n in enumerate(in_names)}\n        self.channels = list(cfg["channels"])\n        self.src = [lookup.get(c.casefold(), -1) for c in self.channels]\n        missing = [c for c, i in zip(self.channels, self.src) if i < 0]\n        print(\n            f"[CovLDA-BCI] matched {len(self.channels) - len(missing)}/{len(self.channels)} "\n            f"training channels by name among {len(in_names)} inputs; "\n            f"zero-filled: {missing if missing else \'none\'}; bands {[b[0] for b in self.bands]}"\n        )\n        if len(missing) == len(self.src):\n            raise RuntimeError(f"no training channel found in the input: {list(in_names)[:8]}")\n        self.reset_state()\n\n    def reset_state(self):\n        C = len(self.channels)\n        self.sum = {name: torch.zeros(C, C, dtype=torch.float64) for name, _, _ in self.bands}\n        self.count = 0\n\n    @torch.no_grad()\n    def predict(self, X):\n        X = torch.as_tensor(X).detach().cpu().double()\n        x = torch.zeros(X.shape[0], len(self.src), X.shape[-1], dtype=torch.float64)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                x[:, dst] = X[:, src]\n        if self.in_sfreq != self.fs:\n            x = resample_fft(x, int(round(x.shape[-1] * self.fs / self.in_sfreq)))\n        out = []\n        for i in range(x.shape[0]):                  # windows are handled in order, one at a time\n            xi = x[i:i + 1]\n            feats = []\n            for name, lo, hi in self.bands:\n                C = oas_cov(band_filter(xi, lo, hi, self.fs))\n                L_ref = (self.n0 * self.prior[name] + self.sum[name]) / (self.n0 + self.count)\n                A = sym_fn(L_ref, lambda w: torch.exp(-w / 2))\n                feats.append(tangent_vec(A @ C @ A))\n                self.sum[name] = self.sum[name] + sym_fn(C, torch.log)[0]\n            self.count += 1\n            out.append((torch.cat(feats, 1) @ self.coef.T + self.intercept).argmax(-1))\n        return torch.cat(out)\n\n\nclass Solver(CompetSolver):\n\n    name = "CovLDA-BCI"\n\n    requirements = []\n\n    def load_model(self, meta):\n        sub = meta["submission_dir"]\n        cfg = json.loads((sub / "covlda_config.json").read_text())\n        if meta.get("n_classes") not in (None, cfg["n_classes"]):\n            raise RuntimeError(\n                f"trained for {cfg[\'n_classes\']} classes, harness asks for {meta[\'n_classes\']}"\n            )\n        in_names = meta.get("ch_names") or [c["ch_name"] for c in meta["chs_info"]]\n        state = torch.load(sub / "covlda_state.pt", map_location="cpu", weights_only=True)\n        return CovLDA(cfg, state, in_names, meta["sfreq"])\n'

In [ ]:
if ZIP_COV and 'band_features' not in globals():
    print('Run Step 3a first (RUN_COV = True): the shipped model reuses its covariance features.')
elif ZIP_COV:
    import copy, importlib.util, shutil, sys, types
    bands = list(FB8)
    tr_all = np.arange(len(SD['Ya']))                     # every window of all 10 people, all sessions
    F_all = torch.cat([band_features(b_, tr_all, 'causal') for b_ in bands], 1).numpy()
    clf = _LDA(solver='lsqr', shrinkage='auto').fit(F_all, SD['Ya'])
    assert list(clf.classes_) == [0, 1, 2]
    train_bal = float(balanced_accuracy_score(SD['Ya'], clf.predict(F_all)))
    out = BASE / 'submissions' / ('track2_covlda_' + '-'.join(bands))
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    state = {'coef': torch.tensor(clf.coef_, dtype=torch.float64),
             'intercept': torch.tensor(clf.intercept_, dtype=torch.float64)}
    for b_ in bands:
        state[f'prior_{b_}'] = LOGCOV[b_][tr_all].mean(0).clone()
    torch.save(state, out / 'covlda_state.pt')
    (out / 'covlda_config.json').write_text(json.dumps(dict(
        channels=list(CH), sfreq=SFREQ, n_classes=N_CLASSES, n0=CLASSICAL_N0, mode='causal',
        bands=[[b_, BAND[b_][0], BAND[b_][1]] for b_ in bands],
        trained_on=f'PROTEUS, {len(set(SD["Pa"]))} people, all sessions, harness default preprocessing',
        train_bal_acc_not_an_estimate=train_bal), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY_COV)

    # Load it the way the harness does, with a stand-in for the base class.
    stub = types.ModuleType('benchmark_utils.base_solver')
    stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
    sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
    sys.modules['benchmark_utils.base_solver'] = stub
    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    def load(names, n_classes=N_CLASSES):
        return mod.Solver().load_model({'ch_names': list(names), 'chs_info': None, 'sfreq': SFREQ,
                                        'submission_dir': out, 'device': 'cpu', 'n_classes': n_classes})

    def stream(model, idx, names=None, extra=0):
        m_ = copy.deepcopy(model); m_.reset_state()      # a fresh copy per session, as in the harness
        res = []
        for i in idx:
            x = SD['XFa'][i:i + 1]
            if names is not None:
                x = x[:, names]
            if extra:
                x = np.concatenate([x, np.random.default_rng(0).standard_normal((1, extra, x.shape[-1])).astype('float32')], 1)
            res.append(int(m_.predict(torch.tensor(x))[0]))
        return np.array(res)

    model0 = load(CH)
    picks = [SESSIONS[0], SESSIONS[len(SESSIONS) // 2], SESSIONS[-1]]
    n_diff = n_tot = 0
    for idx in picks:
        n_diff += int((stream(model0, idx) != clf.predict(F_all[idx])).sum()); n_tot += len(idx)
    first = picks[0][:40]
    base = stream(model0, first)
    perm = np.random.default_rng(1).permutation(len(CH))
    shuffled = stream(load([CH[i] for i in perm]), first, names=perm)
    wide = stream(load(list(CH) + ['EOG1', 'EMG1']), first, extra=2)
    keep_ch = [c for c in CH if c != 'Cz']
    part = stream(load(keep_ch), first, names=[CH.index(c) for c in keep_ch])
    m_a = copy.deepcopy(model0); m_a.reset_state()
    for i in picks[1][:30]:
        m_a.predict(torch.tensor(SD['XFa'][i:i + 1]))
    m_a.reset_state()
    after_reset = np.array([int(m_a.predict(torch.tensor(SD['XFa'][i:i + 1]))[0]) for i in first])
    t0 = time.time(); stream(model0, first[:20]); ms = (time.time() - t0) / 20 * 1000
    try:
        load(CH, n_classes=4); wrong_cls = False
    except RuntimeError:
        wrong_cls = True
    tol = max(1, int(0.005 * n_tot))
    checks = [
        ('labels in {0,1,2}', set(base.tolist()) <= {0, 1, 2}, f'classes {sorted(set(base.tolist()))}'),
        ('same as this notebook (streamed, one window at a time)', n_diff <= tol,
         f'{n_diff} of {n_tot} windows differ over 3 sessions (allowed {tol})'),
        ('order-independent', np.array_equal(base, shuffled), 'electrodes shuffled'),
        ('extra electrodes ignored', np.array_equal(base, wide), 'two unknown channels added'),
        ('missing electrode', part.shape == base.shape, 'Cz zero-filled, still runs'),
        ('reset_state clears the session', np.array_equal(base, after_reset), 'a new session starts fresh'),
        ('wrong class count raises', wrong_cls, 'asked for 4 classes: refuses rather than guessing'),
        ('speed', ms < 200, f'{ms:.0f} ms per window on CPU'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    print(f'  (training balanced accuracy on its own training windows: {train_bal:.3f}; not an estimate)')
    if all(c[1] for c in checks):
        zpath = shutil.make_archive(str(out), 'zip', out)
        print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e3:.0f} KB)')
        print('Download it from Drive and upload to Codabench (Track 2).')
    else:
        print('\nNot zipping: a check failed.')
else:
    print('Step 4a skipped (ZIP_COV = False)')

## Step 4b — package REVE LoRA with causal session scaling

Trains one final model on **all** windows of the 10 people (no held-out data, so the curve shows training loss only), ships the encoder once plus adapters and head, and checks the folder the way the harness runs it: windows one at a time per session with `reset_state`, the same answers as this notebook, electrode order, extra and missing electrodes, wrong class count and speed.

In [ ]:
# Source of truth for the REVE solver: solvers/track2_bci_reve_lora.py in the repo.
SUBMISSION_PY_REVE = '"""Track 02 submission: pretrained REVE with LoRA adapters and a head, averaged over seeds.\n\nThe encoder is `brain-bzh/reve-base`. Its weights ship once, in `encoder.pt`, and stay\nfrozen. Each seed ships only its LoRA adapters (rank r, on the 44 attention projections\n`transformer.layers.*.0.{to_qkv,to_out}`, and optionally a weak adapter on the two\nfeed-forward matrices of each block, `transformer.layers.*.1.net.{1,3}`) and its head, in\n`member_<i>.pt`. The head is per-electrode (Track 1\'s), a pooled-query head, or the plain\naverage, as the config says. The\nmembers share one copy of the encoder in memory, because the harness deep-copies the\nsubmitted model for every session.\n\nInput handling repeats the training notebook: electrodes matched to the training ones by\nname (extras ignored, missing ones zero-filled), the window cropped to the first\n`n_times` samples (the notebook\'s CROP_S), per-window per-channel standardisation, then\nresampling from the harness\'s 120 Hz to REVE\'s 200 Hz. With `input_norm: window` the model\nkeeps no state; with `input_norm: session` the windows are scaled by the running power of their own session (causal;\nstate cleared by `reset_state`). Class probabilities are averaged over members before the\nargmax.\n\nThe model definition matches `track2_03_sealed_eval.ipynb`; keep the two in step.\n"""\n\nimport json\nimport os\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import REVE\n\nREVE_SFREQ = 200.0     # REVE\'s pretraining rate\nPATCH = 200            # REVE\'s patch_size in samples (1 s)\nLORA_TARGETS = ("to_qkv", "to_out")\nFFN_TARGETS = (".net.1", ".net.3")       # FeedForward.net is Sequential(norm, Linear, GEGLU, Linear)\n\n\ndef resample_fft(x, n_out):\n    """Resample along time via the rfft spectrum. Pure torch: nothing is installed at\n    evaluation."""\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\ndef standardize(x):\n    return (x - x.mean(-1, keepdim=True)) / torch.sqrt(\n        x.var(-1, keepdim=True, unbiased=False) + 1e-8\n    )\n\n\nclass LoRALinear(nn.Module):\n    """Frozen base plus a rank-r update. Names (`base`, `A`, `B`) match training."""\n\n    def __init__(self, base, r, alpha):\n        super().__init__()\n        self.base = base\n        self.A = nn.Parameter(torch.zeros(r, base.in_features))\n        self.B = nn.Parameter(torch.zeros(base.out_features, r))\n        self.scale = alpha / r\n\n    def forward(self, x):\n        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale\n\n    @property\n    def weight(self):\n        """Effective weight, W + scale * B A. braindecode >= 2.0 reads `net[1].weight` of the\n        feed-forward GEGLU directly instead of calling the module, which would skip the\n        adapter; with this property the adapter is applied there too. (The input dropout of\n        this adapter does not apply on that path.)"""\n        return self.base.weight + (self.B @ self.A) * self.scale\n\n\nclass PerElectrodeHead(nn.Module):\n    """Shrink each electrode\'s embedding with weights shared across electrodes (one linear\n    layer, or a funnel of several), then a final layer with separate weights per electrode.\n    norm: a LayerNorm after every funnel linear layer, which keeps the signal size steady.\n    init: \'kaiming\' gives the funnel layers He starting weights (PyTorch\'s default shrinks the\n    signal at every layer)."""\n\n    def __init__(self, n_chans, embed_dim, dims, n_outputs, dropout=0.0, mid_dropout=0.0,\n                 norm=False, init=\'default\'):\n        super().__init__()\n        dims = [dims] if isinstance(dims, int) else list(dims)\n        self.norm = nn.LayerNorm(embed_dim)\n        if len(dims) == 1 and not norm:\n            self.shrink = nn.Linear(embed_dim, dims[0])        # keeps the older key names\n        else:\n            layers, prev = [], embed_dim\n            for i, d in enumerate(dims):\n                layers.append(nn.Linear(prev, d))\n                prev = d\n                if norm:\n                    layers.append(nn.LayerNorm(d))\n                if i < len(dims) - 1:\n                    layers += [nn.GELU(), nn.Dropout(mid_dropout)]\n            self.shrink = nn.Sequential(*layers)\n        if init == \'kaiming\':\n            for m in self.shrink.modules():\n                if isinstance(m, nn.Linear):\n                    nn.init.kaiming_normal_(m.weight, nonlinearity=\'relu\')\n                    nn.init.zeros_(m.bias)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(n_chans * dims[-1], n_outputs)\n\n    def forward(self, tokens):                    # (B, chan, embed)\n        z = self.drop(F.gelu(self.shrink(self.norm(tokens))))\n        return self.out(z.flatten(1))\n\n\nclass MQPoolHead(nn.Module):\n    """K learned queries, each pooling the electrodes by a content-weighted average."""\n\n    def __init__(self, k, embed_dim, n_outputs, dropout=0.0):\n        super().__init__()\n        self.queries = nn.Parameter(torch.randn(k, embed_dim) / embed_dim ** 0.5)\n        self.norm = nn.LayerNorm(embed_dim)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(k * embed_dim, n_outputs)\n\n    def forward(self, tokens):                    # (B, chan, embed)\n        z = self.norm(tokens)\n        w = torch.einsum("ke,bce->bkc", self.queries, z) / self.queries.shape[-1] ** 0.5\n        w = w.softmax(dim=-1)\n        pooled = torch.einsum("bkc,bce->bke", w, z)\n        return self.out(self.drop(F.gelu(pooled.flatten(1))))\n\n\ndef is_adapter_key(k):\n    return k.endswith((".A", ".B")) or k.startswith("head.")\n\n\nclass ReveLoRA(nn.Module):\n\n    def __init__(self, channels, n_classes, in_sfreq, rank, encoder_state,\n                 ffn_rank=0, ffn_alpha=0, head_kind="mean", shrink=16, mq_k=2, head_dims=None, head_norm=False):\n        super().__init__()\n        self.head_kind = head_kind\n        self.in_sfreq = float(in_sfreq)\n        self.net = REVE(\n            n_outputs=n_classes, n_chans=len(channels), n_times=PATCH,\n            sfreq=REVE_SFREQ, chs_info=[{"ch_name": c} for c in channels],\n        )\n        self.net.final_layer = nn.Identity()\n        res = self.net.load_state_dict(encoder_state, strict=False)\n        if res.unexpected_keys:\n            raise RuntimeError(f"encoder.pt does not fit REVE: {res.unexpected_keys[:4]}")\n        for p in self.net.parameters():\n            p.requires_grad_(False)\n        names = [\n            n for n, m in self.net.named_modules()\n            if isinstance(m, nn.Linear) and n.split(".")[-1] in LORA_TARGETS\n        ]\n        for name in names:\n            parent = self.net.get_submodule(name.rsplit(".", 1)[0])\n            setattr(parent, name.split(".")[-1],\n                    LoRALinear(self.net.get_submodule(name), rank, 2 * rank))\n        if ffn_rank:                    # the feed-forward matrices, two per block\n            ffn_names = [\n                n for n, m in self.net.named_modules()\n                if isinstance(m, nn.Linear) and n.startswith("transformer.layers.")\n                and n.endswith(FFN_TARGETS)\n            ]\n            for name in ffn_names:\n                parent = self.net.get_submodule(name.rsplit(".", 1)[0])\n                setattr(parent, name.split(".")[-1],\n                        LoRALinear(self.net.get_submodule(name), ffn_rank, ffn_alpha))\n        E = self.net.embed_dim\n        if head_kind == "perelec":\n            self.head = PerElectrodeHead(len(channels), E, head_dims or shrink, n_classes, norm=head_norm)\n        elif head_kind == "mqpool":\n            self.head = MQPoolHead(mq_k, E, n_classes)\n        else:\n            self.head = nn.Linear(E, n_classes)\n\n    def tie_encoder_to(self, other):\n        """Share every frozen tensor with `other`, so ensemble members hold one encoder."""\n        theirs = dict(other.named_parameters())\n        for name, _ in list(self.named_parameters()):\n            if is_adapter_key(name):\n                continue\n            parent = self.get_submodule(name.rsplit(".", 1)[0])\n            setattr(parent, name.rsplit(".", 1)[1], theirs[name])\n\n    def forward(self, x):\n        x = resample_fft(x.float(), int(round(x.shape[-1] * REVE_SFREQ / self.in_sfreq)))\n        feats = self.net(x, return_features=True)["features"]     # (B, chan, patch, embed)\n        if self.head_kind == "mean":\n            return self.head(feats.mean(dim=(1, 2)))\n        return self.head(feats.mean(dim=2))                       # average the time patches only\n\n\nclass NameMatchedEnsemble:\n    """Reorders incoming electrodes into the trained order, zero-filling gaps, and\n    averages the members\' class probabilities."""\n\n    def __init__(self, members, channels, in_names, n_times, device,\n                 input_norm="window", prior_var=None, n0=5.0):\n        self.members, self.device, self.n_times = members, device, int(n_times)\n        self.input_norm, self.n0 = input_norm, float(n0)\n        self.prior_var = None if prior_var is None else torch.as_tensor(prior_var, dtype=torch.float64)\n        lookup = {n.casefold(): i for i, n in enumerate(in_names)}\n        self.src = [lookup.get(c.casefold(), -1) for c in channels]\n        missing = [c for c, i in zip(channels, self.src) if i < 0]\n        print(\n            f"[REVE-LoRA-BCI] matched {len(channels) - len(missing)}/{len(channels)} "\n            f"training channels by name among {len(in_names)} inputs; "\n            f"zero-filled: {missing if missing else \'none\'}; "\n            f"{len(members)} ensemble members"\n        )\n        if len(missing) == len(self.src):\n            raise RuntimeError(\n                f"no training channel found in the input: {list(in_names)[:8]}"\n            )\n        self.reset_state()\n\n    def reset_state(self):\n        # Only the session-scaled model has state: the summed power of the windows seen so far.\n        self.var_sum = torch.zeros(len(self.src), dtype=torch.float64)\n        self.count = 0\n\n    def _session_scale(self, out):\n        """Causal per-session scaling (same as `session_norm` in the training notebook): remove each window\'s\n        mean, divide each electrode by the root of the mean power of the EARLIER windows of the session\n        (blended with a prior of n0 windows), then add this window\'s power to the running sum."""\n        scaled = []\n        for i in range(out.shape[0]):\n            x = out[i].double().cpu()\n            x = x - x.mean(-1, keepdim=True)\n            scale = torch.sqrt((self.n0 * self.prior_var + self.var_sum) / (self.n0 + self.count))\n            scaled.append((x / scale[:, None]).float())\n            self.var_sum = self.var_sum + x.var(-1, unbiased=False)\n            self.count += 1\n        return torch.stack(scaled).to(out.device)\n\n    @torch.no_grad()\n    def predict(self, X):\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        if out.shape[-1] > self.n_times:\n            out = out[..., : self.n_times]\n        elif out.shape[-1] < self.n_times:\n            out = F.pad(out, (0, self.n_times - out.shape[-1]))\n        out = self._session_scale(out) if self.input_norm == "session" else standardize(out)\n        probs = torch.stack([m(out).softmax(-1) for m in self.members]).mean(0)\n        return probs.argmax(-1)\n\n\nclass Solver(CompetSolver):\n\n    name = "REVE-LoRA-BCI"\n\n    requirements = ["pip::braindecode"]\n\n    def load_model(self, meta):\n        sub = meta["submission_dir"]\n        os.environ["REVE_POSITIONS_PATH"] = str(sub)        # bundled; no download at eval\n        cfg = json.loads((sub / "reve_bci_config.json").read_text())\n        if meta.get("n_classes") not in (None, cfg["n_classes"]):\n            raise RuntimeError(\n                f"trained for {cfg[\'n_classes\']} classes, harness asks for "\n                f"{meta[\'n_classes\']}"\n            )\n        in_names = meta.get("ch_names") or [c["ch_name"] for c in meta["chs_info"]]\n        enc_state = torch.load(sub / "encoder.pt", map_location="cpu", weights_only=True)\n        members = []\n        for fname in cfg["member_files"]:\n            m = ReveLoRA(cfg["channels"], cfg["n_classes"], meta["sfreq"],\n                         cfg["rank"], enc_state,\n                         ffn_rank=cfg.get("ffn_rank", 0), ffn_alpha=cfg.get("ffn_alpha", 0),\n                         head_kind=cfg.get("head_kind", "mean"),\n                         shrink=cfg.get("head_shrink", 16), mq_k=cfg.get("mq_k", 2),\n                         head_dims=cfg.get("head_dims"), head_norm=cfg.get("head_norm", False))\n            state = torch.load(sub / fname, map_location="cpu", weights_only=True)\n            res = m.load_state_dict(state, strict=False)\n            absent = [k for k in res.missing_keys if is_adapter_key(k)]\n            if res.unexpected_keys or absent:\n                raise RuntimeError(f"{fname} mismatch: "\n                                   f"{(res.unexpected_keys or absent)[:4]}")\n            if members:\n                m.tie_encoder_to(members[0])\n            members.append(m)\n        members = [m.to(meta["device"]).eval() for m in members]\n        return NameMatchedEnsemble(members, cfg["channels"], in_names,\n                                   cfg["n_times"], meta["device"],\n                                   input_norm=cfg.get("input_norm", "window"),\n                                   prior_var=cfg.get("prior_var"), n0=cfg.get("n0", 5.0))\n'

In [ ]:
if ZIP_REVE:
    import copy, importlib.util, shutil, sys, types
    from braindecode.models import REVE as _REVE
    cat = lambda d: np.concatenate([d[g] for g in ('train', 'val', 'test')])
    Xall, Yall = cat(X), cat(Y)                       # session-scaled windows of all 10 people, all sessions
    keep_before, KEEP = KEEP, 'last'
    finals, prob_fns, curves = [], [], []
    for s_ in range(FINAL_SEEDS):
        # every window is a training window here, so there is no held-out data; the 256 windows passed as
        # "validation" only satisfy the function and mean nothing
        net_s, hist_s, probs_s = train_reve_lora(Xall, Yall, Xall[:256], Yall[:256], EPOCHS,
                                                 f'final fit, seed {s_ + 1}', quiet=True, seed=SEED + s_)
        finals.append(net_s); prob_fns.append(probs_s); curves.append([h['train_loss'] for h in hist_s])
    KEEP = keep_before
    plt.figure(figsize=(5, 3.4))
    for c_ in curves:
        plt.plot(range(1, len(c_) + 1), c_)
    plt.axhline(float(np.log(3)), ls='--', c='grey', lw=1)
    plt.ylim(*PLOT_LOSS_YLIM); plt.grid(alpha=0.3); plt.xlabel('epoch'); plt.ylabel('training loss')
    plt.title('final fit on all 10 people (no held-out data)', fontsize=9); plt.show()

    out = BASE / 'submissions' / 'track2_reve_session'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    os.environ.pop('REVE_POSITIONS_PATH', None)
    enc = _REVE.from_pretrained('brain-bzh/reve-base', n_outputs=N_CLASSES, n_chans=N_CHANS,
                                n_times=200, sfreq=200.0, chs_info=[{'ch_name': c} for c in CH])
    torch.save({k: v.cpu() for k, v in enc.state_dict().items() if not k.startswith('final_layer')}, out / 'encoder.pt')
    del enc
    files = [f'member_{i}.pt' for i in range(len(finals))]
    for f_, m in zip(files, finals):
        torch.save({k: v.cpu() for k, v in m.state_dict().items() if k.endswith(('.A', '.B')) or k.startswith('head.')}, out / f_)
    (out / 'reve_bci_config.json').write_text(json.dumps(dict(
        channels=list(CH), n_times=int(N_TIMES), sfreq=SFREQ, n_classes=N_CLASSES, rank=LORA_RANK,
        ffn_rank=LORA_RANK_FFN, ffn_alpha=LORA_ALPHA_FFN, head_kind='perelec', head_shrink=HEAD_SHRINK,
        head_dims=list(HEAD_DIMS) if HEAD_DIMS else None, head_norm=HEAD_NORM, mq_k=2,
        input_norm='session', prior_var=[float(v) for v in prior_var], n0=5.0,
        member_files=files, crop_s=CROP_S,
        trained_on=f'PROTEUS, {len(set(SD["Pa"]))} people, all sessions, causal session scaling'), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY_REVE)
    import huggingface_hub
    bank = list((Path.home() / 'mne_data').glob('*reve*positions*.json')) or \
        [Path(huggingface_hub.hf_hub_download('brain-bzh/reve-positions', 'positions.json'))]
    shutil.copy(bank[0], out / 'reve_positions.json')

    # Load it the way the harness does, with a stand-in for the base class.
    stub = types.ModuleType('benchmark_utils.base_solver')
    stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
    sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
    sys.modules['benchmark_utils.base_solver'] = stub
    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    def load(names, n_classes=N_CLASSES):
        return mod.Solver().load_model({'ch_names': list(names), 'chs_info': None, 'sfreq': SFREQ,
                                        'submission_dir': out, 'device': DEVICE, 'n_classes': n_classes})

    def stream(model, idx, names=None, extra=0):
        m_ = copy.deepcopy(model); m_.reset_state()      # a fresh copy per session, as in the harness
        res = []
        for i in idx:
            x = SD['XFa'][i:i + 1]                       # the harness sends the full 4 s window
            if names is not None:
                x = x[:, names]
            if extra:
                x = np.concatenate([x, np.random.default_rng(0).standard_normal((1, extra, x.shape[-1])).astype('float32')], 1)
            res.append(int(m_.predict(torch.tensor(x))[0]))
        return np.array(res)

    model0 = load(CH)
    picks = [SESSIONS[0], SESSIONS[len(SESSIONS) // 2], SESSIONS[-1]]
    n_diff = n_tot = 0
    for idx in picks:
        ref = np.mean([pf(SD['Xa'][idx]) for pf in prob_fns], 0).argmax(1)    # this notebook's answer
        n_diff += int((stream(model0, idx) != ref).sum()); n_tot += len(idx)
    first = picks[0][:30]
    base = stream(model0, first)
    perm = np.random.default_rng(1).permutation(len(CH))
    shuffled = stream(load([CH[i] for i in perm]), first, names=perm)
    wide = stream(load(list(CH) + ['EOG1', 'EMG1']), first, extra=2)
    keep_ch = [c for c in CH if c != 'Cz']
    part = stream(load(keep_ch), first, names=[CH.index(c) for c in keep_ch])
    m_b = copy.deepcopy(model0); m_b.reset_state()
    as_batch = m_b.predict(torch.tensor(SD['XFa'][first])).cpu().numpy()      # one call, windows handled in order
    m_a = copy.deepcopy(model0); m_a.reset_state()
    for i in picks[1][:20]:
        m_a.predict(torch.tensor(SD['XFa'][i:i + 1]))
    m_a.reset_state()
    after_reset = np.array([int(m_a.predict(torch.tensor(SD['XFa'][i:i + 1]))[0]) for i in first])
    t0 = time.time(); stream(model0, first[:15]); ms = (time.time() - t0) / 15 * 1000
    try:
        load(CH, n_classes=4); wrong_cls = False
    except RuntimeError:
        wrong_cls = True
    tol = max(1, int(0.01 * n_tot))
    checks = [
        ('labels in {0,1,2}', set(base.tolist()) <= {0, 1, 2}, f'classes {sorted(set(base.tolist()))}'),
        ('same as this notebook (streamed, one window at a time)', n_diff <= tol,
         f'{n_diff} of {n_tot} windows differ over 3 sessions (allowed {tol})'),
        ('one call equals one window at a time', int((as_batch != base).sum()) <= 1, 'same stream, same answers'),
        ('order-independent', np.array_equal(base, shuffled), 'electrodes shuffled'),
        ('extra electrodes ignored', np.array_equal(base, wide), 'two unknown channels added'),
        ('missing electrode', part.shape == base.shape, 'Cz zero-filled, still runs'),
        ('reset_state clears the session', np.array_equal(base, after_reset), 'a new session starts fresh'),
        ('wrong class count raises', wrong_cls, 'asked for 4 classes: refuses rather than guessing'),
        ('speed', ms < 2000, f'{ms:.0f} ms per window on {DEVICE}'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if all(c[1] for c in checks):
        zpath = shutil.make_archive(str(out), 'zip', out)
        print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e6:.0f} MB)')
        print('Download it from Drive and upload to Codabench (Track 2).')
    else:
        print('\nNot zipping: a check failed.')
else:
    print('REVE zip skipped (ZIP_REVE = False)')